# Portfolio Allocation Pie Chart

## Introduction to Creating Interactive Pie Charts with Plotly in Python

Visualizing data effectively is crucial for understanding and interpreting the underlying trends and patterns. Pie charts are a popular way to represent the composition of a dataset, showing the relative proportions of different categories in a visually appealing manner.

In [ ]:
%pip install plotly yfinance -q

In [ ]:
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from collections import defaultdict
from datetime import datetime, timedelta

In [ ]:
# Create your portfolio here
tickers = ['SPY','QQQ']
weights = [0.60,0.40]
portfolio = dict(zip(tickers, weights))
print('Portfolio:', portfolio)

# Remember to change asset classes accoring to your portfolio
asset_classes = dict(zip(tickers, ['USA Stocks','USA Stocks']))

In [ ]:
# Sum weights by asset class
class_weights = defaultdict(float)
for ticker, weight in portfolio.items():
    class_weights[asset_classes[ticker]] += weight

labels = list(class_weights.keys())
sizes = list(class_weights.values())

fig = px.pie(
    names=labels,
    values=sizes,
    title='Portfolio Allocation',
    hole=0.3
)

fig.update_traces(textinfo='percent+label', hoverinfo='label+percent')
fig.show()

## Country Distribution

Fetch country information for each holding using yfinance, then aggregate weighted country exposure across the portfolio.

In [ ]:
# Known ETF country templates (subset of common ETFs)
_KNOWN_ETF_COUNTRIES = {
    'SP500': {'United States': 1.00},
    'MSCI_WORLD': {
        'United States': 0.705, 'Japan': 0.058, 'United Kingdom': 0.038,
        'France': 0.032, 'Canada': 0.030, 'Switzerland': 0.026, 'Germany': 0.023,
        'Australia': 0.019, 'Netherlands': 0.014, 'Denmark': 0.009,
    },
    'MSCI_EM': {
        'China': 0.27, 'India': 0.18, 'Taiwan': 0.18, 'South Korea': 0.12,
        'Brazil': 0.05, 'Saudi Arabia': 0.04, 'Other': 0.16
    },
    'MSCI_ACWI': {
        'United States': 0.635, 'Japan': 0.052, 'United Kingdom': 0.034,
        'France': 0.029, 'Canada': 0.027, 'China': 0.026, 'Other': 0.197
    },
    'EUROPE': {
        'United Kingdom': 0.22, 'France': 0.18, 'Germany': 0.15, 'Switzerland': 0.13,
        'Netherlands': 0.07, 'Sweden': 0.06, 'Denmark': 0.05, 'Other': 0.14
    },
}
_TICKER_TO_TEMPLATE = {
    'SPY': 'SP500', 'VOO': 'SP500', 'IVV': 'SP500', 'QQQ': 'SP500',
    'IWDA': 'MSCI_WORLD', 'SWDA': 'MSCI_WORLD', 'EUNL': 'MSCI_WORLD',
    'VWCE': 'MSCI_ACWI', 'ACWI': 'MSCI_ACWI',
    'EEM': 'MSCI_EM', 'VWO': 'MSCI_EM',
    'VGK': 'EUROPE', 'IEUR': 'EUROPE',
}

def get_country_weights(ticker, weight):
    """Return dict of {country: weighted_exposure}."""
    tmpl = _TICKER_TO_TEMPLATE.get(ticker.upper())
    if tmpl:
        return {c: w * weight for c, w in _KNOWN_ETF_COUNTRIES[tmpl].items()}
    # Try yfinance
    try:
        info = yf.Ticker(ticker).info
        country = info.get('country')
        if country:
            return {country: weight}
        raw = info.get('countryWeightings') or []
        cw = {}
        for d in raw:
            for k, v in d.items():
                cw[k] = cw.get(k, 0) + float(v)
        if cw:
            return {c: v * weight for c, v in cw.items()}
    except Exception:
        pass
    return {'Unknown': weight}

# Aggregate country exposure
country_agg = defaultdict(float)
for ticker, wt in portfolio.items():
    cw = get_country_weights(ticker, wt)
    for country, val in cw.items():
        country_agg[country] += val

# Sort and display
country_df = sorted(country_agg.items(), key=lambda x: -x[1])
print('Country Exposure:')
for c, w in country_df:
    print(f'  {c}: {w:.1%}')

In [ ]:
# Plot country distribution bar chart
countries = [c for c, _ in country_df]
exposures = [w * 100 for _, w in country_df]

fig = px.bar(
    x=exposures, y=countries,
    orientation='h',
    labels={'x': 'Exposure (%)', 'y': 'Country'},
    title='Portfolio Country Distribution',
    text=[f'{e:.1f}%' for e in exposures]
)
fig.update_layout(yaxis=dict(autorange='reversed'), height=max(400, len(countries) * 28 + 100))
fig.show()

## Sector Distribution

Fetch sector weights for each holding and aggregate portfolio-level sector exposure.

In [ ]:
# Known ETF sector templates
_KNOWN_ETF_SECTORS = {
    'SP500': {
        'Information Technology': 0.29, 'Financials': 0.13, 'Health Care': 0.13,
        'Consumer Discretionary': 0.10, 'Industrials': 0.09,
        'Communication Services': 0.09, 'Consumer Staples': 0.06,
        'Energy': 0.04, 'Materials': 0.02, 'Real Estate': 0.02, 'Utilities': 0.02,
    },
    'MSCI_WORLD': {
        'Information Technology': 0.24, 'Financials': 0.16, 'Health Care': 0.12,
        'Industrials': 0.10, 'Consumer Discretionary': 0.10,
        'Communication Services': 0.08, 'Consumer Staples': 0.07,
        'Energy': 0.05, 'Materials': 0.04, 'Real Estate': 0.02, 'Utilities': 0.02,
    },
    'MSCI_EM': {
        'Information Technology': 0.22, 'Financials': 0.22, 'Consumer Discretionary': 0.13,
        'Communication Services': 0.10, 'Materials': 0.09, 'Energy': 0.07,
        'Industrials': 0.06, 'Consumer Staples': 0.05, 'Health Care': 0.04, 'Utilities': 0.02,
    },
    'EUROPE': {
        'Financials': 0.18, 'Industrials': 0.16, 'Health Care': 0.15,
        'Consumer Staples': 0.12, 'Consumer Discretionary': 0.10,
        'Materials': 0.08, 'Energy': 0.07, 'Information Technology': 0.07,
        'Utilities': 0.04, 'Communication Services': 0.03,
    },
    'MSCI_ACWI': {
        'Information Technology': 0.24, 'Financials': 0.16, 'Health Care': 0.11,
        'Industrials': 0.10, 'Consumer Discretionary': 0.10,
        'Communication Services': 0.08, 'Consumer Staples': 0.07,
        'Energy': 0.05, 'Materials': 0.04, 'Real Estate': 0.02, 'Utilities': 0.02,
    },
}

_SECTOR_NORM = {
    'Financial Services': 'Financials', 'Technology': 'Information Technology',
    'Healthcare': 'Health Care', 'Realestate': 'Real Estate',
    'Communication': 'Communication Services', 'Consumer Cyclical': 'Consumer Discretionary',
    'Consumer Defensive': 'Consumer Staples', 'Basic Materials': 'Materials',
}

def get_sector_weights(ticker, weight):
    """Return dict of {sector: weighted_exposure}."""
    tmpl = _TICKER_TO_TEMPLATE.get(ticker.upper())
    if tmpl and tmpl in _KNOWN_ETF_SECTORS:
        return {s: w * weight for s, w in _KNOWN_ETF_SECTORS[tmpl].items()}
    try:
        info = yf.Ticker(ticker).info
        sector = info.get('sector')
        if sector:
            canon = _SECTOR_NORM.get(sector, sector)
            return {canon: weight}
        raw = info.get('sectorWeightings') or []
        sw = {}
        for d in raw:
            for k, v in d.items():
                canon = _SECTOR_NORM.get(k.title(), k.title())
                sw[canon] = sw.get(canon, 0) + float(v)
        if sw:
            return {s: v * weight for s, v in sw.items()}
    except Exception:
        pass
    return {'Other': weight}

sector_agg = defaultdict(float)
for ticker, wt in portfolio.items():
    sw = get_sector_weights(ticker, wt)
    for sector, val in sw.items():
        sector_agg[sector] += val

sector_df = sorted(sector_agg.items(), key=lambda x: -x[1])
print('Sector Exposure:')
for s, w in sector_df:
    print(f'  {s}: {w:.1%}')

In [ ]:
# Sector colours
SECTOR_COLORS = {
    'Information Technology': '#1a78c2', 'Health Care': '#e63946',
    'Financials': '#e6a817', 'Consumer Discretionary': '#f4631e',
    'Consumer Staples': '#f5c518', 'Energy': '#b85c00',
    'Industrials': '#5b8db8', 'Materials': '#a0522d',
    'Real Estate': '#c8a96e', 'Communication Services': '#7c4dff',
    'Utilities': '#80c41c', 'Other': '#888888',
}

sectors = [s for s, _ in sector_df]
s_exposures = [w * 100 for _, w in sector_df]
s_colors = [SECTOR_COLORS.get(s, '#888888') for s in sectors]

fig = px.bar(
    x=s_exposures, y=sectors,
    orientation='h',
    labels={'x': 'Exposure (%)', 'y': 'Sector'},
    title='Portfolio Sector Distribution',
    text=[f'{e:.1f}%' for e in s_exposures],
    color=sectors,
    color_discrete_map=SECTOR_COLORS
)
fig.update_layout(yaxis=dict(autorange='reversed'), showlegend=False,
                  height=max(400, len(sectors) * 30 + 100))
fig.show()